## CDF JSON RAG

Main points:
- Meaningful split of chunks semantically:
  - Event rows → 1 event per chunk.
  - Meta and Sheet data → JSON chunks split by meaningful subtrees:
    - `meta`
    - `sheet.match.result`
    - `sheet.events.goals`
    - `sheet.events.substitutions`
    - `sheet.events.cards`
    - `sheet.teams.home`
    - `sheet.teams.away`

In [9]:
from pathlib import Path
# pip install chromadb requests openai
from typing import List, Dict, Any, Optional, Iterable, Tuple
import json
import os

import requests
import chromadb
from chromadb.config import Settings
from openai import OpenAI


project_root = Path(".").resolve()
combined_path = project_root / "cdf_all_matches.json"

def _json_dumps(obj: Any) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False)
    except TypeError:
        return json.dumps(str(obj), ensure_ascii=False)


def _split_text_into_chunks(text: str, max_len: int) -> List[str]:
    if max_len <= 0 or len(text) <= max_len:
        return [text]
    return [text[i:i + max_len] for i in range(0, len(text), max_len)]

# LLM (Open WebUI, OpenAI-compatible) — same defaults as kg_triples_rag_grouped_by_subject.ipynb
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "https://web.ollama-gpt-oss.ai.wu.ac.at/api")
LLM_API_KEY = os.getenv("LLM_API_KEY", "sk-9b41b856cc0b403b8a3c10618f1c2996")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-oss:120b")

llm_client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    timeout=120.0,
    max_retries=4,
)

# ---- Chroma (persistent) ----
OLLAMA_EMBED_BASE = os.getenv("OLLAMA_EMBED_BASE", "http://localhost:11434")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
CHROMA_PATH = project_root / "chroma_db_cdf_json"
COLLECTION_NAME = "cdf_records_raw_semantic"
RESET_COLLECTION = False
SKIP_IF_COMPLETE = True
BATCH_SIZE = 500


In [10]:
def _is_event_row(obj: Dict[str, Any]) -> bool:
    return isinstance(obj, dict) and ("event_id" in obj) and ("event_time" in obj) and ("event_type" in obj)


def _is_match_container(obj: Dict[str, Any]) -> bool:
    return isinstance(obj, dict) and ("meta" in obj) and ("sheet" in obj)


def build_event_chunks_from_row(event_row: Dict[str, Any], *, max_raw_chars: int = 1500) -> List[Dict[str, Any]]:
    # Building chunks by also adding an ID to each chunk
    match_id = str(event_row.get("match_id", "unknown"))
    event_id = str(event_row.get("event_id", "unknown"))

    payload_text = f"RAW_JSON match_id={match_id} section=event item={event_id}\n" + _json_dumps(event_row)

    chunks: List[Dict[str, Any]] = []
    for seg_idx, segment in enumerate(_split_text_into_chunks(payload_text, max_raw_chars)):
        chunks.append(
            {
                "id": f"{match_id}::event::{event_id}::seg::{seg_idx}",
                "text": segment,
            }
        )

    return chunks


In [11]:
def _raw_json_chunks(match_id: str, section: str, payload: Any, *, max_raw_chars: int) -> List[Dict[str, Any]]:
    # One subtree => raw JSON chunk.
    # Output chunks contain only: {id, text}.
    payload_text = f"RAW_JSON match_id={match_id} section={section}\n" + _json_dumps(payload)
    out: List[Dict[str, Any]] = []
    for seg_idx, segment in enumerate(_split_text_into_chunks(payload_text, max_raw_chars)):
        out.append(
            {
                "id": f"{match_id}::{section}::seg::{seg_idx}",
                "text": segment,
            }
        )
    return out


def _raw_json_items(match_id: str, section: str, items: List[Any], *, max_raw_chars: int) -> List[Dict[str, Any]]:
    # Chunking for if they exceed max_raw_chars, so they are split in the segments
    out: List[Dict[str, Any]] = []
    for i, item in enumerate(items):
        payload_text = f"RAW_JSON match_id={match_id} section={section} item={i}\n" + _json_dumps(item)
        for seg_idx, segment in enumerate(_split_text_into_chunks(payload_text, max_raw_chars)):
            out.append(
                {
                    "id": f"{match_id}::{section}::item{i}::seg::{seg_idx}",
                    "text": segment,
                }
            )
    return out


def build_match_container_chunks(
    match_obj: Dict[str, Any],
    *,
    max_raw_chars: int = 1500,
    include_events: bool = True,
) -> List[Dict[str, Any]]:
    # Chunk a match container (has `meta` and `sheet`) into raw JSON subtrees only.
    match_id = str(
        match_obj.get("match_id")
        or match_obj.get("meta", {}).get("match_id")
        or match_obj.get("sheet", {}).get("match_id")
        or "unknown"
    )

    meta = match_obj.get("meta", {}) or {}
    sheet = match_obj.get("sheet", {}) or {}

    chunks: List[Dict[str, Any]] = []

    # Core subtrees
    chunks.extend(_raw_json_chunks(match_id, "meta", meta, max_raw_chars=max_raw_chars))

    sheet_match = (sheet.get("match", {}) or {})
    result = (sheet_match.get("result", {}) or {})
    chunks.extend(
        _raw_json_chunks(
            match_id,
            "match status",
            (sheet_match.get("status", {}) or {}),
            max_raw_chars=max_raw_chars,
        )
    )
    chunks.extend(_raw_json_chunks(match_id, "match result", result, max_raw_chars=max_raw_chars))

    # sheet.events split into per-item chunks
    sheet_events = (sheet.get("events", {}) or {})
    chunks.extend(_raw_json_items(match_id, "goal", sheet_events.get("goals", []) or [], max_raw_chars=max_raw_chars))
    chunks.extend(
        _raw_json_items(
            match_id,
            "substitution",
            sheet_events.get("substitutions", []) or [],
            max_raw_chars=max_raw_chars,
        )
    )
    chunks.extend(_raw_json_items(match_id, "sheet card", sheet_events.get("cards", []) or [], max_raw_chars=max_raw_chars))

    # sheet.teams: split into home/away subtrees (stable football match structure)
    sheet_teams = (sheet.get("teams", {}) or {})
    chunks.extend(_raw_json_chunks(match_id, "team home", (sheet_teams.get("home", {}) or {}), max_raw_chars=max_raw_chars))
    chunks.extend(_raw_json_chunks(match_id, "team away", (sheet_teams.get("away", {}) or {}), max_raw_chars=max_raw_chars))

    # referees as their own subtree
    chunks.extend(_raw_json_chunks(match_id, "referees", sheet.get("referees", []) or [], max_raw_chars=max_raw_chars))

    # tracking events from event_cdf.json (nested under match["events"])
    if include_events:
        for event_row in match_obj.get("events") or []:
            if _is_event_row(event_row):
                chunks.extend(build_event_chunks_from_row(event_row, max_raw_chars=max_raw_chars))

    return chunks


In [12]:
def build_chunks_from_record(
    record: Dict[str, Any],
    *,
    max_raw_chars: int = 1500,
    include_events: bool = True,
) -> List[Dict[str, Any]]:
    # Route a record to the appropriate raw-JSON chunker, where
    # - event rows (have event_id/event_time/event_type)
    # - match containers (have meta + sheet)

    if _is_event_row(record):
        if not include_events:
            return []
        return build_event_chunks_from_row(record, max_raw_chars=max_raw_chars)

    if _is_match_container(record):
        return build_match_container_chunks(
            record,
            max_raw_chars=max_raw_chars,
            include_events=include_events,
        )

    return []


def load_and_chunk_records(
    combined_json_path: Path,
    *,
    max_records: Optional[int] = 2000,
    only_match_id: Optional[str] = None,
    max_raw_chars: int = 1500,
    include_events: bool = True,
) -> List[Dict[str, Any]]:
    # Load `cdf_all_matches.json` and convert records into raw JSON chunks.
    # Can't embed chunks larger than 1500 characters.
    # With models like "mxbai-embed-large" this number is even smaller.
    with open(combined_json_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if only_match_id is not None:
        records = [r for r in records if str(r.get("match_id")) == str(only_match_id)]

    if max_records is not None:
        records = records[:max_records]

    all_chunks: List[Dict[str, Any]] = []
    for rec in records:
        all_chunks.extend(
            build_chunks_from_record(
                rec,
                max_raw_chars=max_raw_chars,
                include_events=include_events,
            )
        )

    print("Records loaded:", len(records))
    print("Include tracking events:", include_events)
    print("Chunks generated:", len(all_chunks))
    if all_chunks:
        print("Example chunk:")
        print("- id:", all_chunks[0]["id"])
        print("- text (first 300 chars):\n", all_chunks[0]["text"][:300])

    return all_chunks


# include_events=False -> sheet/meta only (~928 chunks for 34 matches)
# include_events=True  -> adds tracking events from event_cdf (~138k chunks)
chunked_documents = load_and_chunk_records(
    combined_path,
    max_records=2000,
    max_raw_chars=1500,
    include_events=True,
)


Records loaded: 34
Include tracking events: True
Chunks generated: 138353
Example chunk:
- id: 3895052::meta::seg::0
- text (first 300 chars):
 RAW_JSON match_id=3895052 section=meta
{"competition": {"competition_id": 9, "competition_name": "1. Bundesliga"}, "season_id": 281, "match_id": "3895052", "match_kickoff_time": "2023-08-19T16:30:00.000Z", "match": {"periods": [{"type": "first half", "play_direction": "right left"}, {"type": "second


In [13]:
import json
from pathlib import Path

out_path = project_root / "docs_I.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(chunked_documents, f, ensure_ascii=False, indent=2)

print(f"Wrote {len(chunked_documents)} chunks to {out_path}")

Wrote 138353 chunks to C:\Users\dyury\Desktop\Master Thesis\docs_I.json


In [14]:
class OllamaEmbeddingFunction:
    def __init__(self, model: str = EMBED_MODEL, base_url: str = OLLAMA_EMBED_BASE):
         self.model = model
         self.url = f"{base_url}/api/embeddings"

    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        vectors: List[List[float]] = []
        for text in texts:
            response = requests.post(
                self.url,
                json={"model": self.model, "prompt": text},
                timeout=120,
            )
            response.raise_for_status()
            data = response.json()
            vectors.append(data["embedding"])
        return vectors

    def embed_documents(self, input: List[str]) -> List[List[float]]:
        return self._embed_texts(input)

    def embed_query(self, input):
        if isinstance(input, str):
            return self._embed_texts([input])[0]
        return self._embed_texts(input)

    def __call__(self, input: List[str]) -> List[List[float]]:
        return self.embed_documents(input)

    def name(self) -> str:
        return f"ollama-{self.model}"


embedding_function = OllamaEmbeddingFunction()
CHROMA_PATH.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))

if RESET_COLLECTION:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted collection:", COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_function,
)

ids_all = [doc["id"] for doc in chunked_documents]
docs_all = [doc["text"] for doc in chunked_documents]
existing_count = collection.count()
print(f"Chroma count before indexing: {existing_count} / {len(ids_all)}")

if SKIP_IF_COMPLETE and existing_count >= len(ids_all) and len(ids_all) > 0:
    print("Skipping indexing: collection already complete.")
else:
    existing_ids = set(collection.get(include=[])["ids"])
    pending = [i for i, doc_id in enumerate(ids_all) if doc_id not in existing_ids]
    if not pending:
        print("Skipping indexing: all doc ids already in Chroma.")
    else:
        print(f"Indexing {len(pending)} new docs ({len(existing_ids)} already present)")
        for batch_start in range(0, len(pending), BATCH_SIZE):
            batch_idx = pending[batch_start : batch_start + BATCH_SIZE]
            collection.add(
                ids=[ids_all[i] for i in batch_idx],
                documents=[docs_all[i] for i in batch_idx],
            )
            print(f"  Indexed {min(batch_start + BATCH_SIZE, len(pending))}/{len(pending)} new")

print("Chroma count after indexing:", collection.count())


Chroma count before indexing: 138353 / 138353
Skipping indexing: collection already complete.
Chroma count after indexing: 138353


In [15]:
def call_llm(prompt: str, model: str = LLM_MODEL) -> str:
    resp = llm_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return (resp.choices[0].message.content or "").strip()


def cdf_json_rag_qa(
    question: str,
    top_k: int = 12,
    show_retrieved: bool = False,
    preview_chars: int = 300,
) -> dict:
    results = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents"],
    )
    retrieved_ids = results.get("ids", [[]])[0]
    retrieved_docs = results.get("documents", [[]])[0]

    if show_retrieved:
        print(f"Question: {question}")
        print("Retrieved chunks")
        print("-" * 100)
        for i, doc in enumerate(retrieved_docs):
            preview = (doc[:preview_chars] + "...") if isinstance(doc, str) and len(doc) > preview_chars else doc
            print(f"[{i}]")
            print(preview)
            print("-" * 100)

    context = "\n\n".join(retrieved_docs)
    prompt = f"""
You are a helpful assistant answering questions about football matches in the CDF dataset.

Use ONLY the information in the context below. If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}

Answer clearly and briefly.
""".strip()

    answer = call_llm(prompt)
    return {
        "question": question,
        "approach": "cdf_json_rag",
        "retrieved_ids": retrieved_ids,
        "contexts": list(retrieved_docs),
        "answer": answer,
        "error": None,
    }


def rag_answer(question: str, top_k: int = 12, show_retrieved: bool = False, preview_chars: int = 300) -> str:
    return cdf_json_rag_qa(question, top_k=top_k, show_retrieved=show_retrieved, preview_chars=preview_chars)["answer"]



In [ ]:
# Example
question = "What is the final score for match 3895052."
print(rag_answer(question, top_k=10, show_retrieved=True))


Question: What is the final score for match 3895052.
Retrieved chunks
----------------------------------------------------------------------------------------------------
[0]
RAW_JSON match_id=3895052 section=event item=d3ce943e-c1c8-40dd-8ac7-5ee4f614a750
{"match_id": "3895052", "meta": {"is synced": false}, "event_id": "d3ce943e-c1c8-40dd-8ac7-5ee4f614a750", "event_time": "00:46:29.439", "event_period": "second half", "event_type": "ball receipt*", "event_sub_type": nu...
----------------------------------------------------------------------------------------------------
[1]
RAW_JSON match_id=3895113 section=event item=f3fdcac6-8e4e-4785-bd28-06d9fb49093c
{"match_id": "3895113", "meta": {"is synced": false}, "event_id": "f3fdcac6-8e4e-4785-bd28-06d9fb49093c", "event_time": "01:18:31.558", "event_period": "second half", "event_type": "ball receipt*", "event_sub_type": nu...
----------------------------------------------------------------------------------------------------
[2]
RAW_JSO

In [ ]:
# Example
question = "What is the final score for match Bayer Leverkusen against RB Leipzig."
print(rag_answer(question, top_k=10, show_retrieved=True))


Question: What is the final score for match Bayer Leverkusen against RB Leipzig.
Retrieved chunks
----------------------------------------------------------------------------------------------------
[0]
RAW_JSON match_id=3895052 section=match result
{"final": {"home": 3, "away": 2, "winning_team_id": "904"}, "first_half": {"home": 2, "away": 1}, "second_half": {"home": 1, "away": 1}, "first_half_extratime": {"home": 0, "away": 0}, "second_half_extratime": {"home": 0, "away": 0}, "shootout": {"home"...
----------------------------------------------------------------------------------------------------
[1]
RAW_JSON match_id=3895052 section=goal item=0
{"time": "00:23:04.554", "player_id": "32712", "assist_id": "32289", "team_id": "904", "is_own_goal": false, "is_penalty": false, "score": {"home": 1, "away": 0}}
----------------------------------------------------------------------------------------------------
[2]
RAW_JSON match_id=3895052 section=goal item=1
{"time": "00:34:03.174", "pl

In [ ]:
# Example
question = "What teams played in match 3895052."
print(rag_answer(question, top_k=10))


Based on the result section of the data, the teams that played are:

* Home team: Team with ID 904 (won)
* Away team: Team with ID 182


In [ ]:
question = "How many matches were played in the tournament."
print(rag_answer(question, top_k=40))

There were **26 matches** recorded in the tournament.


In [ ]:
## Batch eval — questions from `qa_eval_questions.txt`

## Results merge into `qa_eval_cdf_json_rag_outputs.jsonl` (includes `contexts` for Ragas).

In [ ]:
questions_path = project_root / "qa_eval_questions.txt"
output_path = project_root / "qa_eval_cdf_json_rag_outputs.jsonl"

QUESTION_START = 1
QUESTION_END = None

all_questions = [
    line.strip()
    for line in questions_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

start_idx = max((QUESTION_START or 1) - 1, 0)
end_idx = QUESTION_END if QUESTION_END is not None else len(all_questions)
questions = all_questions[start_idx:end_idx]
question_indices = list(range(start_idx + 1, end_idx + 1))

print(f"Loaded {len(all_questions)} questions")
print(f"Output: {output_path}")


def load_existing_records(path: Path) -> dict[int, dict]:
    records: dict[int, dict] = {}
    if not path.exists():
        return records
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        q_idx = record.get("question_index")
        if q_idx is None and record.get("question") in all_questions:
            q_idx = all_questions.index(record["question"]) + 1
        if q_idx is None:
            q_idx = -(line_no)
        records[int(q_idx)] = record
    return records


def run_cdf_record(question: str, question_index: int) -> dict:
    record = {
        "question_index": question_index,
        "question": question,
        "approach": "cdf_json_rag",
        "retrieved_ids": None,
        "contexts": None,
        "answer": None,
        "error": None,
    }
    try:
        out = cdf_json_rag_qa(question, top_k=12, show_retrieved=False)
        record["retrieved_ids"] = out.get("retrieved_ids")
        record["contexts"] = out.get("contexts") or []
        record["answer"] = out.get("answer")
    except Exception as e:
        record["error"] = str(e)
    return record


existing_records = load_existing_records(output_path)
batch_results: list[dict] = []

for q_idx, question in zip(question_indices, questions):
    print(f"\n[{q_idx}/{len(all_questions)}] {question[:100]}")
    record = run_cdf_record(question, q_idx)
    existing_records[q_idx] = record
    batch_results.append(record)
    if record["error"]:
        print("  ERROR:", record["error"])
    else:
        print("  ANSWER:", (record["answer"] or "")[:120])

with output_path.open("w", encoding="utf-8") as f:
    for q_idx in sorted(existing_records):
        f.write(json.dumps(existing_records[q_idx], ensure_ascii=False) + "\n")

ok = sum(1 for r in batch_results if not r["error"])
print(f"\nDone. {ok}/{len(batch_results)} ok; {len(existing_records)} total in file.")